# Enterprise Agentic RAG — Evaluation

Runs the RAG pipeline against a labeled evaluation set and computes:
- **RAGAS metrics**: faithfulness, answer relevancy, context precision, context recall
- **Retrieval-only metrics**: precision@k, recall@k, MRR

Make sure `.env` is configured (`GROQ_API_KEY`, `HF_TOKEN`) and Weaviate is running before executing this notebook.

In [ ]:
# Add project root to path so `src` imports resolve when running from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.evaluation.evaluation_dataset import load_eval_dataset, save_eval_dataset
from src.evaluation.ragas_eval import run_evaluation
from src.evaluation.retrieval_metrics import precision_at_k, recall_at_k, mean_reciprocal_rank
from src.graph.rag_graph import run_rag_pipeline

## 1. Load (or create) a labeled evaluation dataset
Each sample needs a `question` and a `ground_truth` answer. Optionally add `relevant_chunk_ids` for retrieval-only metrics.

In [ ]:
eval_samples = load_eval_dataset()

# If none exist yet, seed a small example set (replace with real labeled data).
if not eval_samples:
    eval_samples = [
        {
            "question": "What is our company's refund policy?",
            "ground_truth": "Refunds are issued within 30 days of purchase with a valid receipt.",
            "relevant_chunk_ids": [],
        }
    ]
    save_eval_dataset(eval_samples)

eval_samples

## 2. Run RAGAS evaluation (faithfulness, relevancy, context precision/recall)

In [ ]:
ragas_results = run_evaluation(eval_samples)
ragas_results

## 3. Retrieval-only metrics (precision@k / recall@k / MRR)
Requires each sample to have `relevant_chunk_ids` labeled ahead of time.

In [ ]:
K = 5
for sample in eval_samples:
    if not sample.get("relevant_chunk_ids"):
        continue
    state = run_rag_pipeline(sample["question"])
    retrieved_ids = [c["metadata"].get("chunk_id") for c in state.get("chunks", [])]
    p = precision_at_k(retrieved_ids, sample["relevant_chunk_ids"], K)
    r = recall_at_k(retrieved_ids, sample["relevant_chunk_ids"], K)
    mrr = mean_reciprocal_rank(retrieved_ids, sample["relevant_chunk_ids"])
    print(f"Q: {sample['question']}\n  precision@{K}={p:.2f}  recall@{K}={r:.2f}  MRR={mrr:.2f}\n")